# Experiment 1: Does reranking improve retrieval? (RAG retrieval evaluation)

**Your name / date:** ______

## Meaning
In RAG, step 1 is *retrieval*: find the passages that may answer a question. Step 2 can be *reranking*: a second, smarter model re-orders those passages so the best one comes first. The LLM only sees the top few, so order matters.

## Why it came into the picture
Fast retrieval (keywords or embeddings) often finds the right passage but puts it at rank 3 or 5. A cross-encoder reads the question and each passage together, so it judges better, but it is slower. So we retrieve a few candidates cheaply, then rerank only those.

## What we do here (no paid API, no key, runs on your laptop)
1. 12 small support passages and 10 questions. For each question we know the correct passage.
2. **Before:** a simple keyword retriever (plain Python) gives its top 5.
3. **After:** a CrossEncoder re-orders those 5.
4. We measure Hit@1, MRR and time, and look at which questions improved or failed.

Rule for interviews: only claim the numbers **you** get when you run this. Write them in the last cell.

## Step 0: Install (run once in PowerShell, then restart the kernel)

`pip install sentence-transformers`

The first run downloads a small model (about 90 MB) from Hugging Face. It needs internet only that one time.

## Step 1: The documents and questions
Each document has an id. Each question has the id of the passage that really answers it (`expected`).

In [1]:
documents = {
    "d1": (
        "To reset your password, open Settings, choose Security, then "
        "click Forgot password. A reset link is emailed to you and "
        "expires after 30 minutes."
    ),
    "d2": (
        "Refunds are available within 14 days of purchase. Contact "
        "billing with your order number and the money returns to the "
        "original card in 5 to 7 business days."
    ),
    "d3": (
        "You can change your subscription plan at any time from the "
        "Billing page. An upgrade starts immediately and a downgrade "
        "starts at the next billing cycle."
    ),
    "d4": (
        "Two-factor authentication adds a second check when you sign "
        "in. Turn it on under Settings, Security, and scan the QR code "
        "with an authenticator app."
    ),
    "d5": (
        "To delete your account, go to Settings, Account, and choose "
        "Delete account. All your data is removed permanently after 7 "
        "days and cannot be restored."
    ),
    "d6": (
        "Our support team answers email and chat from 9 AM to 6 PM IST, "
        "Monday to Friday. Messages sent on weekends are answered on "
        "Monday."
    ),
    "d7": (
        "Invoices are generated on the first day of every month. "
        "Download them from the Billing page in PDF format for your "
        "accounting records."
    ),
    "d8": (
        "If the app crashes on start, clear the cache, update to the "
        "latest version and restart your phone. If it still fails, send "
        "us the crash report."
    ),
    "d9": (
        "Team admins can invite new members from the Members page by "
        "entering their email addresses. Invited people get a link that "
        "works for 7 days."
    ),
    "d10": (
        "You can export all your projects as a ZIP file from Settings, "
        "Data. The export is emailed to you within 24 hours."
    ),
    "d11": (
        "Notifications can be turned off per project. Open the project, "
        "click the bell icon and choose Mute. You will still get "
        "billing emails."
    ),
    "d12": (
        "We store your data in encrypted form on servers in India. "
        "Backups are kept for 30 days and then deleted automatically."
    ),
}

# question, expected document id
questions = [
    ("How do I get my money back after buying?", "d2"),
    ("I forgot my login secret, what should I do?", "d1"),
    ("How can I make my sign in safer?", "d4"),
    ("Where can I download my monthly bill?", "d7"),
    ("What happens when I move to a cheaper plan?", "d3"),
    ("How do I stop getting alerts from a project?", "d11"),
    ("Can I take a copy of all my work out of the app?", "d10"),
    ("When can I talk to someone on the support team?", "d6"),
    ("The application keeps closing when I open it, help", "d8"),
    ("How do I add a colleague to my team?", "d9"),
]
print(len(documents), "documents,", len(questions), "questions")

12 documents, 10 questions


## Step 2: BEFORE. A simple keyword retriever
It counts how many words the question and the passage share (ignoring tiny common words). More shared words means a higher score. It does not understand meaning, so questions that use different words than the passage (like *money back* vs *refunds*) can fail. That is the point of the experiment.

In [2]:
import time

STOP_WORDS = ["the", "a", "an", "is", "are", "to", "of", "in", "on", "i", "my", "do", "how",
              "can", "you", "your", "and", "or", "it", "for", "what", "when", "where", "me",
              "we", "our", "at", "from", "with", "will", "if", "get", "after"]

def tokenize(text):
    text = text.lower()
    cleaned = ""
    for ch in text:
        if ch.isalnum() or ch == " ":
            cleaned = cleaned + ch
        else:
            cleaned = cleaned + " "
    words = []
    for w in cleaned.split():
        if w not in STOP_WORDS:
            words.append(w)
    return words

def keyword_search(question, top_k=5):
    q_words = tokenize(question)
    scored = []
    for doc_id in documents:
        d_words = tokenize(documents[doc_id])
        score = 0
        for w in q_words:
            if w in d_words:
                score = score + 1
        scored.append((doc_id, score))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    top_ids = []
    for pair in scored[:top_k]:
        top_ids.append(pair[0])
    return top_ids

print(keyword_search("How do I get my money back after buying?"))

['d2', 'd1', 'd3', 'd4', 'd5']


## Step 3: AFTER. Rerank the same 5 candidates with a CrossEncoder
The CrossEncoder reads (question, passage) pairs and gives each a relevance score. We sort by that score. Note: it only re-orders the 5 candidates, so if the right passage is not in the 5, reranking cannot save it.

In [3]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question, candidate_ids):
    pairs = []
    for doc_id in candidate_ids:
        pairs.append((question, documents[doc_id]))
    scores = model.predict(pairs)
    combined = []
    for i in range(len(candidate_ids)):
        combined.append((candidate_ids[i], float(scores[i])))
    combined.sort(key=lambda pair: pair[1], reverse=True)
    ordered_ids = []
    for pair in combined:
        ordered_ids.append(pair[0])
    return ordered_ids

rerank("How do I get my money back after buying?", keyword_search("How do I get my money back after buying?"))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

['d2', 'd5', 'd1', 'd3', 'd4']

## Step 4: Run both on every question and time them
Rank = position of the correct passage (1 is best). If it is not in the top 5 we use 0 for 'missing'.

In [4]:
def rank_of(expected, result_ids):
    position = 1
    for doc_id in result_ids:
        if doc_id == expected:
            return position
        position = position + 1
    return 0

# warm up the model once so the first call does not distort timing
rerank("warm up", ["d1", "d2"])

rows = []
before_time = 0.0
after_time = 0.0
for question, expected in questions:
    t0 = time.perf_counter()
    before_ids = keyword_search(question, top_k=5)
    t1 = time.perf_counter()
    after_ids = rerank(question, before_ids)
    t2 = time.perf_counter()
    before_time = before_time + (t1 - t0)
    after_time = after_time + (t2 - t1)
    rows.append({
        "question": question,
        "expected": expected,
        "rank_before": rank_of(expected, before_ids),
        "rank_after": rank_of(expected, after_ids),
    })

for r in rows:
    print(r["expected"], "before:", r["rank_before"], "after:", r["rank_after"])
    print("   question:", r["question"])

d2 before: 1 after: 1
   question: How do I get my money back after buying?
d1 before: 1 after: 1
   question: I forgot my login secret, what should I do?
d4 before: 1 after: 1
   question: How can I make my sign in safer?
d7 before: 1 after: 1
   question: Where can I download my monthly bill?
d3 before: 1 after: 1
   question: What happens when I move to a cheaper plan?
d11 before: 1 after: 1
   question: How do I stop getting alerts from a project?
d10 before: 4 after: 1
   question: Can I take a copy of all my work out of the app?
d6 before: 1 after: 1
   question: When can I talk to someone on the support team?
d8 before: 0 after: 0
   question: The application keeps closing when I open it, help
d9 before: 2 after: 1
   question: How do I add a colleague to my team?


## Step 5: The numbers
**Hit@1** = in how many questions the correct passage is first. **MRR** = average of 1/rank (higher is better, max 1.0).

In [5]:
def hit_at_1(ranks):
    count = 0
    for r in ranks:
        if r == 1:
            count = count + 1
    return count / len(ranks)

def mrr(ranks):
    total = 0.0
    for r in ranks:
        if r > 0:
            total = total + 1 / r
    return total / len(ranks)

before_ranks = []
after_ranks = []
for r in rows:
    before_ranks.append(r["rank_before"])
    after_ranks.append(r["rank_after"])
n = len(rows)

before_hit = round(hit_at_1(before_ranks), 2)
after_hit = round(hit_at_1(after_ranks), 2)
before_mrr = round(mrr(before_ranks), 2)
after_mrr = round(mrr(after_ranks), 2)
print("BEFORE (keyword)  Hit@1:", before_hit, " MRR:", before_mrr)
print("AFTER  (rerank)   Hit@1:", after_hit, " MRR:", after_mrr)

before_ms = round(before_time / n * 1000, 2)
after_ms = round(after_time / n * 1000, 2)
print("Avg time per question, keyword:", before_ms, "ms")
print("Avg time per question, rerank :", after_ms, "ms")

improved = []
worse = []
still_failed = []
for r in rows:
    # rank 0 means "not found", so treat it as a very bad rank (99)
    b_score = r["rank_before"]
    if b_score == 0:
        b_score = 99
    a_score = r["rank_after"]
    if a_score == 0:
        a_score = 99
    if a_score < b_score:
        improved.append(r["question"])
    elif a_score > b_score:
        worse.append(r["question"])
    if a_score != 1:
        still_failed.append(r["question"])

print()
print("Improved:", len(improved))
for q in improved:
    print("  +", q)
print("Got worse:", len(worse))
for q in worse:
    print("  -", q)
print("Not rank 1 after reranking:", len(still_failed))
for q in still_failed:
    print("  x", q)

BEFORE (keyword)  Hit@1: 0.7  MRR: 0.78
AFTER  (rerank)   Hit@1: 0.9  MRR: 0.9
Avg time per question, keyword: 0.68 ms
Avg time per question, rerank : 70.94 ms

Improved: 2
  + Can I take a copy of all my work out of the app?
  + How do I add a colleague to my team?
Got worse: 0
Not rank 1 after reranking: 1
  x The application keeps closing when I open it, help


## Step 6: Add Groq to generate the answer (this part needs your free Groq key)
Retrieval and reranking decide **which passages** the LLM sees. Groq's LLM then writes the **answer** from those passages. We keep these two parts separate on purpose:
- Steps 1 to 5 measured **retrieval** (was the right passage found and ranked first?).
- Step 6 measures **answer quality** (did the answer contain the right fact?). A good retrieval does not guarantee a good answer, and the other way round.

### Key safety (important)
- Never paste your key in this notebook or in WhatsApp. This notebook never prints your key.
- Create a file named `.env` in your repo folder `E:\genai-hands-on-lab`, next to this notebook, with one line: `GROQ_API_KEY=your_key_here`
- The `.gitignore` already in your repo root ignores `.env`, so git will not upload it. `.env.example` is only a placeholder file with a fake value.
- Install: `pip install groq python-dotenv`

Model: `openai/gpt-oss-20b` was listed on Groq's models page on 13 Oct 2026. Model names change, so if you get a "model not found" error, check https://console.groq.com/docs/models and change `GROQ_MODEL` below. Groq's rate-limits page (Free Plan Limits tab) listed this model at 30 requests/min, 1,000 requests/day, 8K tokens/min, 200K tokens/day, so the loop waits a few seconds between calls. When you run the cell below, it asks you once: type `yes` to call Groq, or anything else to skip. There is nothing to edit, so no True/False spelling to get wrong. Stay on Groq's free usage; the limits above are the free ones.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
GROQ_MODEL = "openai/gpt-oss-20b"
key_value = os.environ.get("GROQ_API_KEY", "")
key_found = len(key_value.strip()) > 0
print("Working folder:", os.getcwd())
print("Key loaded from .env:", key_found)

GROQ_ENABLED = False
if not key_found:
    print("No key found, so Groq stays OFF. Steps 1 to 5 still work.")
else:
    question_text = "Type yes to call Groq, anything else skips: "
    answer = input(question_text)
    if answer.strip().lower() == "yes":
        GROQ_ENABLED = True
        print("Groq is ON for this run.")
    else:
        print("Groq stays OFF. Steps 1 to 5 still work.")

Working folder: e:\genai-hands-on-lab
Key loaded from .env: True


Facts that a correct answer must contain. We check the answer text for this phrase (simple and not perfect: a right answer worded differently would be marked wrong, so also read the answers yourself).

In [ ]:
expected_facts = {
    "d2": (
        "14 days"
    ),
    "d1": (
        "30 minutes"
    ),
    "d4": (
        "authenticator"
    ),
    "d7": (
        "first day"
    ),
    "d3": (
        "next billing cycle"
    ),
    "d11": (
        "mute"
    ),
    "d10": (
        "24 hours"
    ),
    "d6": (
        "9 am"
    ),
    "d8": (
        "cache"
    ),
    "d9": (
        "7 days"
    ),
}

def build_prompt(question, passage_ids):
    context = ""
    for doc_id in passage_ids:
        context = context + "- " + documents[doc_id] + "\n"
    prompt = (
        "Answer the question using ONLY the passages below. "
        "If the passages do not contain the answer, reply exactly: I don't know.\n\n"
        "Passages:\n" + context + "\nQuestion: " + question
    )
    return prompt

In [ ]:
if GROQ_ENABLED:
    from groq import Groq, RateLimitError, AuthenticationError, NotFoundError
    client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_groq(prompt):
    # waits and retries ONLY for 'too many requests'. Any other error (wrong key, wrong model name) stops here so you can read it.
    for attempt in range(4):
        try:
            reply = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=2000,
                temperature=0.0,
            )
            text = reply.choices[0].message.content
            if text is None:
                text = ""
            return text
        except AuthenticationError:
            raise RuntimeError("Groq rejected the key. Check GROQ_API_KEY in .env (no quotes, no spaces), or create a new key.")
        except NotFoundError:
            raise RuntimeError("Groq does not know this model name. Check https://console.groq.com/docs/models and change GROQ_MODEL.")
        except RateLimitError:
            print("Rate limit reached - waiting 20 seconds")
            time.sleep(20)
    raise RuntimeError("Still rate limited after 4 tries. Wait a few minutes and re-run this cell.")

def contains_fact(answer, expected_id):
    return expected_facts[expected_id] in answer.lower()

answer_rows = []
if GROQ_ENABLED:
    for question, expected in questions:
        before_ids = keyword_search(question, top_k=5)
        after_ids = rerank(question, before_ids)

        # the LLM only sees the TOP 3 passages from each method
        ans_before = ask_groq(build_prompt(question, before_ids[:3]))
        time.sleep(10)
        ans_after = ask_groq(build_prompt(question, after_ids[:3]))
        time.sleep(10)

        answer_rows.append({
            "question": question,
            "expected": expected,
            "answer_before": ans_before,
            "answer_after": ans_after,
            "fact_before": contains_fact(ans_before, expected),
            "fact_after": contains_fact(ans_after, expected),
        })
        print("done:", question)

In [ ]:
if GROQ_ENABLED and len(answer_rows) > 0:
    good_before = 0
    good_after = 0
    for r in answer_rows:
        if r["fact_before"]:
            good_before = good_before + 1
        if r["fact_after"]:
            good_after = good_after + 1
        print("Q:", r["question"])
        print("  before (keyword top 3):", r["answer_before"].strip()[:200])
        print("     has right fact:", r["fact_before"])
        print("  after  (rerank top 3):", r["answer_after"].strip()[:200])
        print("     has right fact:", r["fact_after"])
        print()
    print("Answers containing the right fact, before:", good_before, "of", len(answer_rows))
    print("Answers containing the right fact, after :", good_after, "of", len(answer_rows))

## Step 7: Your turn (this is what makes it your evidence)
1. Pick one question that improved in Step 5. Did the Groq answer also improve? If not, why not?
2. Look at the question `The application keeps closing...`. What did the LLM say when the right passage was missing? Did it say "I don't know" or did it make something up? This is the difference between a safe and an unsafe RAG answer.
3. Change `top_k=5` to `top_k=3` in Step 4 and compare Hit@1 again. What happens?
4. Add 3 questions of your own (with expected ids and a fact) and re-run.

## Step 8: Write your results (fill after running, use only your real numbers)

**Retrieval (Steps 1 to 5)**

| | Keyword (before) | Reranked (after) |
|---|---|---|
| Hit@1 | | |
| MRR | | |
| Avg time per question (ms) | | |

**Generated answers (Step 6, model used: ______ )**

| | Keyword top 3 | Reranked top 3 |
|---|---|---|
| Answers containing the right fact (out of 10) | | |

**Questions that improved:**

**Questions that failed or got worse, and why:**

**What I learned (3 lines, in my own words):**

**Limits (tiny data: 12 passages, 10 questions, passages written by Instinct; fact check is a simple text match; one run only):**

### How to say it in an interview
"I built a small RAG evaluation with 12 support passages and 10 questions with known answers. I compared a keyword baseline with a CrossEncoder reranker on the same top 5 candidates. Hit@1 went from ___ to ___ and reranking added about ___ ms per question. Then I used a Groq-hosted LLM to answer from the top 3 passages and checked whether the answer contained the right fact: ___ of 10 before and ___ of 10 after. It is a small test, so I treat it as a demonstration of how I measure retrieval separately from answer quality, not as a benchmark."